# 30 — Family stratification

Panel-averaged BEDROC on N=8 targets can hide family-level heterogeneity. Claim A ("no MD-guided lift over GBSA-locked") is stated for the full panel. Split by family, two families have n ≥ 2 (proteases, kinases) and three are singletons. This notebook computes per-family panel BEDROC (+ bootstrap CI) for the four principal rankers, does the same for the top-5 single MD features (from `single_feature_bedroc.csv`), and reframes Claim A.

**Small-n caveat.** Only 2 families reach n ≥ 2 in the discovery set (proteases n=4, kinases n=2). The 18 validation targets would lift kinases to n=10. Family-stratified claims are pilot-level until validation is computed.

> **Bootstrap regime.** This notebook uses B=1000, seed 20250901 for speed. Canonical (see `data/derived/canonical_baselines.csv`) is B=5000, seed 20260902. Numbers here are stable at reported precision; ranges are a hair wider.

> **Reader guide.** *Experiment A3 (see [STUDY_DESIGN §A3](../../STUDY_DESIGN.md)):* per-complex
> MD-feature analysis and downstream ranking questions.
>
> See STUDY_DESIGN Chapter §A3 Q1 (per-target combo selection) and Q2 (single-feature panel
> ranker) for the framing this notebook addresses.
>
> **Reproducibility contract:** reads `data/derived/features.parquet` +
> `data/raw/reference/ohds_metadata.csv` (and `data/derived/canonical_baselines.csv` for
> baseline comparison).

In [ ]:
# --- notebook preamble ---
NB_STEM = "44_family_stratification"

import sys, os, json
from pathlib import Path

# Make the in-repo src package importable without an install
# find repo root robustly (walks up until pyproject.toml)
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / 'pyproject.toml').is_file():
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from discovery9.style import apply_style, NAVY, GOLD, GREY, GREY_DASH as GREYD, CREAM, WHITE, TARGET_COLORS
from discovery9.paths import ROOT, DERIVED, FIGURES, GBSA_STUDY
from discovery9.io import load_features, load_metadata, load_gbsa
from discovery9.metrics import bedroc
apply_style()

# --- fig-capture hook (iter-3 fix) ---
_SAVED_FIGS = globals().setdefault('_SAVED_FIGS', [])
_orig_figure = plt.figure
_orig_subplots = plt.subplots
def _figure_capture(*a, **kw):
    fig = _orig_figure(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig
def _subplots_capture(*a, **kw):
    fig, ax = _orig_subplots(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig, ax
plt.figure = _figure_capture
plt.subplots = _subplots_capture

# Target-family map (from R4 review + upstream ChEMBL classification).
TARGET_FAMILY = {
    '2XU3': 'protease',
    '4A5S': 'protease',
    '4L7G': 'protease',
    '9SI4': 'protease',
    '5HU9': 'kinase',
    '8ELC': 'kinase',
    '4QB3': 'bromodomain',
    '9D9I': 'chaperone',
    '3I06': 'other',
}
FAMILY_ORDER = ['protease', 'kinase', 'bromodomain', 'chaperone', 'other']
FAMILY_N_MIN = 2  # families with fewer than this get flagged as "singleton — not stratifiable"

RNG_SEED = 20250901
BOOT_B = 1000
ALPHA = 20.0

# Custom family colour map (kept inside the NAVY/GOLD/GREY palette)
FAMILY_COLORS = {
    'protease':     NAVY,
    'kinase':       GOLD,
    'bromodomain':  "#7A8AB5",
    'chaperone':    "#B89A3A",
    'other':        GREYD,
}


## 1. Load per-target BEDROC for the four principal rankers

We take the P1 rows from `deep_research_wide.csv` (per-target combo-selection sweep) for:
- **GBSA-locked** (baseline)
- **Ridge**
- **RandomForest**
- **SVM-RBF**

These are the rankers used to state Claim A in NB 23 / 25. Per-target BEDROC values are parsed out of the `per_target` string column (format: `target=value;target=value;…`).

In [ ]:
# Load and parse per-target BEDROC from deep_research_wide.csv
wide = pd.read_csv(DERIVED / 'deep_research_wide.csv')
print(f'deep_research_wide.csv: {len(wide)} model rows across protocols {sorted(wide.protocol.unique())}')

def parse_per_target(s: str) -> dict[str, float]:
    out = {}
    for tok in str(s).split(';'):
        tok = tok.strip()
        if '=' not in tok:
            continue
        k, v = tok.split('=', 1)
        try:
            out[k] = float(v)
        except ValueError:
            pass
    return out

MODELS_OF_INTEREST = ['GBSA-locked', 'Ridge', 'RandomForest', 'SVM-RBF']
sub = wide[(wide.protocol == 'P1') & (wide.model.isin(MODELS_OF_INTEREST))].copy()
if sub.empty:
    # P1 rows are what NB 11 uses; fall back to P2 if we can't find P1 rows for one of the models
    print('WARNING: no P1 rows for the requested models — falling back to whatever protocol exists')
    sub = wide[wide.model.isin(MODELS_OF_INTEREST)].drop_duplicates('model', keep='first').copy()

rows = []
for _, r in sub.iterrows():
    d = parse_per_target(r.per_target)
    for tgt, b in d.items():
        rows.append({'model': r.model, 'target': tgt, 'bedroc': b,
                     'family': TARGET_FAMILY.get(tgt, 'unknown')})
per_t = pd.DataFrame(rows)
print(f'\nPer-target BEDROC table: {len(per_t)} rows')
print(per_t.pivot(index='target', columns='model', values='bedroc').round(3).to_string())
print()
print('Family assignment:')
print(pd.Series(TARGET_FAMILY, name='family').to_string())


## 2. Per-family panel BEDROC + bootstrap CI

For each (family, model) we compute mean BEDROC over targets in that family and the 95% bootstrap CI on the mean (resample targets within family with replacement, B=1000, seed 20250901). Families with fewer than `FAMILY_N_MIN=2` targets are flagged as singletons — we report the point estimate but skip the bootstrap (CI would be degenerate).

In [ ]:
# Per-family panel BEDROC + bootstrap CI (resample targets within family)
def boot_mean(vals, B=BOOT_B, seed=RNG_SEED):
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) < 2:
        return (float(np.nanmean(vals)) if len(vals) else np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(vals), size=(B, len(vals)))
    boots = vals[idx].mean(axis=1)
    return float(vals.mean()), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

rows = []
for fam in FAMILY_ORDER:
    fam_targets = [t for t, f in TARGET_FAMILY.items() if f == fam]
    for model in MODELS_OF_INTEREST:
        vals = per_t[(per_t.family == fam) & (per_t.model == model)].bedroc.values
        mean, lo, hi = boot_mean(vals)
        rows.append({
            'family': fam,
            'n_targets': len(fam_targets),
            'model': model,
            'panel_bedroc': mean,
            'ci_lo': lo, 'ci_hi': hi,
            'stratifiable': len(fam_targets) >= FAMILY_N_MIN,
        })
fam_tbl = pd.DataFrame(rows)
print('Per-family panel BEDROC + 95% bootstrap CI (B=1000, resample targets within family):')
print(fam_tbl.round(3).to_string(index=False))

# Bar chart: per-family panel BEDROC across the four models, grouped by family
# iter-4 FIX (Fig 1 layout): increase bottom margin, rotate x-tick labels 30°,
# move n-annotations ABOVE the axes so they never collide with x-tick labels.
fams_plot   = FAMILY_ORDER
model_order = MODELS_OF_INTEREST
mtx     = fam_tbl.pivot(index='family', columns='model', values='panel_bedroc').reindex(fams_plot)[model_order]
mtx_lo  = fam_tbl.pivot(index='family', columns='model', values='ci_lo').reindex(fams_plot)[model_order]
mtx_hi  = fam_tbl.pivot(index='family', columns='model', values='ci_hi').reindex(fams_plot)[model_order]

fig, ax = plt.subplots(figsize=(11, 5.8))
x = np.arange(len(fams_plot)); w = 0.19
palette = {MODELS_OF_INTEREST[0]: NAVY, MODELS_OF_INTEREST[1]: GOLD,
           MODELS_OF_INTEREST[2]: '#7A8AB5', MODELS_OF_INTEREST[3]: '#B89A3A'}
for i, m in enumerate(model_order):
    means = mtx[m].values
    los   = mtx_lo[m].values
    his   = mtx_hi[m].values
    err_lo = np.where(np.isfinite(los), np.clip(means - los, 0, None), 0)
    err_hi = np.where(np.isfinite(his), np.clip(his - means, 0, None), 0)
    ax.bar(x + (i - len(model_order)/2 + 0.5)*w, means, w, color=palette[m], edgecolor=NAVY,
           linewidth=0.5, yerr=[err_lo, err_hi], capsize=3, ecolor=GREYD, label=m)

# iter-4 FIX: n-annotations moved ABOVE the axis; singleton labels also above.
# X-tick labels rotated 30° for readability. Reserve headroom via ylim.
ax.set_ylim(0, 1.15)
for xi, fam in zip(x, fams_plot):
    n_f = sum(1 for t, f in TARGET_FAMILY.items() if f == fam)
    ax.text(xi, 1.03, f'n={n_f}', ha='center', va='bottom', fontsize=9, color=NAVY,
            transform=ax.get_xaxis_transform())
    if n_f < FAMILY_N_MIN:
        ax.text(xi, 1.09, 'singleton (no CI)', ha='center', va='bottom', fontsize=8,
                color='#a02020', transform=ax.get_xaxis_transform())

ax.set_xticks(x)
ax.set_xticklabels(fams_plot, rotation=30, ha='right')
ax.set_ylabel('per-family panel BEDROC α=20')
ax.set_title('Per-family panel BEDROC for GBSA-locked, Ridge, RandomForest, SVM-RBF\n'
             '(error bars = 95% bootstrap CI; singleton families have no CI)')
ax.legend(fontsize=9, loc='upper right')
ax.set_axisbelow(True); ax.yaxis.grid(True, color=GREY, alpha=0.5)
# iter-4 FIX: extra bottom margin so rotated labels + xlabel line have room.
plt.subplots_adjust(bottom=0.20)
plt.tight_layout()

# (fig auto-captured by preamble hook)


## 3. Top-5 single MD features — per-family BEDROC

Take the top-5 single MD features from `single_feature_bedroc.csv` (ranked by panel BEDROC on the naive 8-target panel), recompute per-target BEDROC on the recovered 9-target panel (4A5S labels restored), then aggregate per family with bootstrap CI as above.

In [ ]:
# Load single-feature ranking table
sf = pd.read_csv(DERIVED / 'single_feature_bedroc.csv')
print(f'single_feature_bedroc.csv: {len(sf)} features')
top5 = sf[~sf.is_lig_chem].head(5).feature.tolist()  # top-5 among MD features
print(f'Top-5 MD features (by naive panel BEDROC): {top5}')

# Rebuild per-target BEDROC on the 9-target panel with 4A5S recovered
meta = load_metadata()
feat = load_features(with_ligand_chem=True)
lab = feat.merge(meta[['complex_id','target','is_active']],
                 on=['complex_id','target'], how='left', suffixes=('','_meta'))
if 'is_active_meta' in lab.columns:
    lab['is_active'] = (lab['is_active'].astype('boolean')
                        .combine_first(lab['is_active_meta'].astype('boolean')))
lab = lab.dropna(subset=['is_active']).copy()
lab['is_active'] = lab['is_active'].astype(bool).astype(int)

# For each feature, use the sign that maximises panel BEDROC on the 9-target panel
def best_sign_perT(f):
    per_pos, per_neg = [], []
    for t, g in lab.groupby('target'):
        y = g.is_active.astype(int).values
        v = g[f].values.astype(float)
        per_pos.append(bedroc(+v, y, alpha=ALPHA))
        per_neg.append(bedroc(-v, y, alpha=ALPHA))
    mp = float(np.nanmean(per_pos)); mn = float(np.nanmean(per_neg))
    return (+1, mp, per_pos) if mp >= mn else (-1, mn, per_neg)

single_rows = []
for f in top5:
    sign, panel, per_t_vals = best_sign_perT(f)
    tgts = sorted(lab.target.unique())
    for t, b in zip(tgts, per_t_vals):
        single_rows.append({'feature': f, 'sign': sign, 'target': t, 'bedroc': b,
                            'family': TARGET_FAMILY.get(t, 'unknown')})
single_per_t = pd.DataFrame(single_rows)
print()
print('Per-target BEDROC for top-5 MD features (best-sign, 9-target panel):')
print(single_per_t.pivot(index='target', columns='feature', values='bedroc').round(3).to_string())

# Per-family aggregate + bootstrap CI
rows = []
for f in top5:
    for fam in FAMILY_ORDER:
        vals = single_per_t[(single_per_t.feature == f) & (single_per_t.family == fam)].bedroc.values
        mean, lo, hi = boot_mean(vals)
        rows.append({'feature': f, 'family': fam,
                     'n_targets': sum(1 for t, ff in TARGET_FAMILY.items() if ff == fam),
                     'panel_bedroc': mean, 'ci_lo': lo, 'ci_hi': hi})
single_fam_tbl = pd.DataFrame(rows)
print()
print('Per-family panel BEDROC for top-5 MD features (95% bootstrap CI, B=1000):')
print(single_fam_tbl.round(3).to_string(index=False))

# Small-multiples chart: one row per family, top-5 features on x-axis
fams_plot = [f for f in FAMILY_ORDER if sum(1 for t, ff in TARGET_FAMILY.items() if ff == f) >= 1]
n_rows = len(fams_plot)
fig, axes = plt.subplots(n_rows, 1, figsize=(10, 2.0 * n_rows), sharex=True)
if n_rows == 1: axes = [axes]
for ax, fam in zip(axes, fams_plot):
    sub = single_fam_tbl[single_fam_tbl.family == fam].set_index('feature').reindex(top5)
    means = sub.panel_bedroc.values
    los   = sub.ci_lo.values
    his   = sub.ci_hi.values
    err_lo = np.where(np.isfinite(los), np.clip(means - los, 0, None), 0)
    err_hi = np.where(np.isfinite(his), np.clip(his - means, 0, None), 0)
    n_f = sub.n_targets.iloc[0] if len(sub) else 0
    color = FAMILY_COLORS.get(fam, GREYD)
    ax.bar(np.arange(len(top5)), means, width=0.55, color=color, edgecolor=NAVY, linewidth=0.5,
           yerr=[err_lo, err_hi], capsize=3, ecolor=GREYD)
    ax.set_title(f'family = {fam}   (n={n_f})', fontsize=10, loc='left')
    ax.set_ylim(0, 1.0)
    ax.axhline(0.5, color=GREY, ls=':', lw=1)
    ax.set_ylabel('panel BEDROC α=20', fontsize=9)
    ax.set_axisbelow(True); ax.yaxis.grid(True, color=GREY, alpha=0.4)
axes[-1].set_xticks(np.arange(len(top5)))
axes[-1].set_xticklabels(top5, rotation=15, ha='right', fontsize=9)
plt.tight_layout()

# (fig auto-captured by preamble hook)


## 4. Verdict — reframing Claim A per family

- **Proteases (n = 4: 2XU3, 4A5S, 4L7G, 9SI4).** All four models cluster near the family mean (see the "protease" bar group above). No model has a CI lower bound above the GBSA-locked CI upper bound. **Claim A holds on the protease family.**
- **Kinases (n = 2: 5HU9, 8ELC).** Both are already at the top of the achievable BEDROC range (GBSA-locked family mean ≈ 0.82). No headroom for ML lift here; observed ML BEDROCs sit within CI of GBSA-locked. **Claim A holds on the kinase family by ceiling.**
- **Bromodomain (n = 1: 4QB3), chaperone (n = 1: 9D9I), other (n = 1: 3I06).** Singletons — no bootstrap CI. We can't rule out per-family ML lift for any of the three. Point estimate only.

**Claim A — reframed.** "No MD-guided panel-BEDROC lift over GBSA-locked" holds for the two families with n ≥ 2. For the three singleton families we can't generalise; needs a larger panel.

**Per-family plateau numbers** (identical across the four principal rankers — GBSA-locked / Ridge / RF / SVM-RBF — because ML picks the same combo as GBSA-locked when LOTO training has no same-family targets):

| Family | n | Panel BEDROC (any of the 4 rankers) | 95% CI |
|--------|---|-------------------------------------|--------|
| protease | 4 | **0.351** | [0.121, 0.581] |
| kinase   | 2 | **0.820** | [0.655, 0.985] |
| bromodomain | 1 | 0.517 (GBSA-locked); 0.132-0.256 (ML) | — |
| chaperone   | 1 | 0.741 | — |
| other       | 1 | 0.568 | — |

<!-- canonical baseline — see data/derived/canonical_baselines.csv -->

Both families with n ≥ 2 plateau — proteases low (≈ 0.35), kinases high (≈ 0.82). That means the panel-averaged canonical GBSA-locked = **0.541** (9T, 4A5S imputed at 0; 0.6088 on the 8-target GBSA subset in the family table) hides a bimodal outcome, not an uncertain moderate mean. The single-feature top-5 chart (§3) makes the same point: the best single MD feature reaches BEDROC ≈ 0.9 on kinases and ≈ 0.4 on proteases.

### 4QB3 collapse — no cherry-picking per target

Most striking per-target signal in Table §1: all three ML models drop from 0.517 (GBSA-locked) to 0.256 on 4QB3 — a full 0.26 collapse on the sole bromodomain in the panel. Every non-GBSA model (Ridge, RandomForest, SVM-RBF) picks the same wrong combo for 4QB3 under LOTO because the training set (8 non-bromodomain targets) has no signal for what makes a good GBSA combo on bromodomains. The models default to the mean of training predictions, which happens to select a combo that scores poorly on this target.

Generic failure mode: you can't cherry-pick a good ML per-target answer if the family isn't represented in training. Bromodomain has n=1 here, so LOTO on 4QB3 gets zero same-family training data. The 18-target validation set has 3 more bromodomains — should let us test whether ML recovers the GBSA-locked baseline once the family is represented.

Take-away for practitioners: single-target BEDROC values from LOTO on a small panel are noisy proxies for real per-target performance. Report family-stratified panel means with CIs; if a single target diverges by > 0.2 from its family peers, flag it as a "no training data for this family" artefact, not a real signal.

**Caveats:**
- 9 targets across 5 families → 3 singleton families. Any per-family conclusion for a singleton is a point estimate, not a claim.
- Family boundaries follow ChEMBL classification; alternative taxonomies (fold family, active-site geometry) would give different groupings.
- The 18-target locked-parameter validation set includes multiple kinases, proteases, and bromodomains; that data will tighten per-family CIs and directly test the 4QB3 hypothesis above.

In [ ]:
# --- export every figure produced in this notebook (iter-3 fix) ---
try:
    FIGURES.mkdir(parents=True, exist_ok=True)
except NameError:
    from discovery9.paths import FIGURES
    FIGURES.mkdir(parents=True, exist_ok=True)
try:
    _cream = CREAM
except NameError:
    from discovery9.style import CREAM as _cream
figs = list(globals().get('_SAVED_FIGS', []))
for num in plt.get_fignums():
    f = plt.figure(num)
    if f not in figs:
        figs.append(f)
saved = []
for i, fig in enumerate(figs, start=1):
    out = FIGURES / f"{NB_STEM}_fig{i}.png"
    try:
        fig.savefig(out, bbox_inches='tight', dpi=300, facecolor=_cream)
    except Exception as e:
        print(f'  WARN: failed to save fig{i}: {e}')
        continue
    saved.append(str(out.name))
print(f'saved {len(saved)} figures:')
for s in saved:
    print(' ', s)


## Where this leaves us — end of pilot

This is the end of the discovery-9 arc. Everything above is the pilot; the next step is the 18-target validation cohort that will either confirm or break these findings.

**What this pilot tells us.** Claim A survives, but only in scoped form: on the 9-target discovery panel, no ML combo-selection pipeline we tested beats GBSA-locked at panel level, and the two families with n ≥ 2 both plateau — proteases at BEDROC ≈ 0.35, kinases at ≈ 0.82. Claim B is retracted: the naive single-feature top on the recovered 9-target panel, `lig_buried_sasa_std_A2` at 0.603 [0.39, 0.78] (was 0.674 on the older 8-target subset that dropped 4A5S), sits inside the GBSA-locked CI [0.36, 0.71] and collapses under every hardening condition. The baseline hierarchy docking → GBSA → MD is a flat null on n = 9: 0.516 / 0.541 / ~0.60, all CIs overlap. The MD → GBSA surrogate has a clean r ≈ 0.455 [0.236, 0.674] once we strip GBSA-energy inputs from the feature set — modest, and nowhere near the r ≈ 0.93 an earlier draft reported (that number was inflated by leaking GBSA components into the feature list).

**What this pilot does not tell us.** Everything family-stratified is pilot-level. Three of the five families here are singletons — bromodomain, chaperone, and one "other". Singletons have no CI, so their point estimates are anecdotes. Kinases sit at n = 2 in discovery; the 18-target validation cohort (pre-registered in `data/external/gbsa-study/data/raw/newbench_targets.csv`, `split == 'validation'`) has 8 more kinases, which would lift the family to n = 10 and finally make the kinase-ceiling claim testable. Proteases would move from n = 4 to n = 5. Every other family in discovery is a singleton and would reach n ≥ 2 with validation added — enough to draw a first CI, but still small. Until those 18 targets are computed, family-stratified statements here are hypotheses, not findings.

**What we would do next.** Three things, in priority order.

1. Compute MD + MM-GBSA for the 18 validation targets under the locked recipe (`igb2_di4_salt0.15_st0.0072`). Wallclock estimate: ~540 productions × ~24 h GPU, plus GBSA rescoring at ~50 s/complex. Schedulable in one compute allocation. Delivers CIs for all 9 families and a proper external test of Claim A.
2. Confirmatory triplicate replicas from the BO winner config (Tier-3 workspace at `/mnt/netapp1/…/discovery9_winner/`, 8 × 30 × 3 reps × 20 ns) to check that the 9-target BEDROC numbers reproduce under replica noise. Pins down whether per-target values are seed-limited or physics-limited.
3. DUD-E MK14 external validation for the 8ELC pocket if the validation-18 cohort doesn't include a comparable kinase (skip if it does). Tests whether the 8ELC BEDROC ≈ 0.985 is a pocket-specific ceiling or a size-confounded artefact.

**What the reader should walk away with.** The retracted Claim B is a case study in how much a single-feature claim can drift between "novel deployable ranker" and "point estimate inside CI" once you (a) recover the missing target labels and (b) apply seven hardening conditions. The scoped Claim A is a modest but genuine null on the panel level. The next iteration will either confirm the null on n = 27 or force a re-scope; either way, the docking / GBSA-lock / MD-feature comparison is now on solid enough footing to run that test.

That's the end of the discovery-9 pilot. See `README.md` and `STUDY_DESIGN.md` for headline numbers and the "what we do not claim" list; `VALIDATION_PLAN.md` for the pre-registered validation cohort; `docs/GLOSSARY.md` for terminology and column definitions.